In [ ]:
# ================================
# CELL 1 — Install
# ================================
!pip install sentence-transformers -q
print("✅ Done!")

In [ ]:
# ================================
# CELL 2 — Check GPU
# ================================
import torch
print(f"GPU available : {torch.cuda.is_available()}")
print(f"GPU name      : "
      f"{torch.cuda.get_device_name(0)}")

In [ ]:
# ================================
# CELL 3 — Load Data
# ================================
import pickle
from google.colab import files

print("Upload processed_data.pkl")
files.upload()

with open('processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

statute_texts   = data['statute_texts']
statute_ids     = data['statute_ids']
precedent_texts = data['precedent_texts']
precedent_ids   = data['precedent_ids']
query_texts     = data['query_texts']
query_ids       = data['query_ids']
gold_statutes   = data['gold_statutes']
gold_precedents = data['gold_precedents']

print(f"✅ Data loaded!")

In [ ]:
# ================================
# CELL 4 — Prepare Clean Texts
# ================================
def prepare_text(text, max_words=400):
    if isinstance(text, list):
        text = ' '.join(text)
    words = text.split()[:max_words]
    return ' '.join(words)

statute_texts_clean   = [prepare_text(t)
    for t in statute_texts]
precedent_texts_clean = [prepare_text(t)
    for t in precedent_texts]
query_texts_clean     = [prepare_text(t)
    for t in query_texts]

print("✅ Clean texts ready!")

In [ ]:
# ================================
# CELL 5 — Encode All Documents
# ================================
from sentence_transformers import SentenceTransformer
import torch

print("Loading model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
model = model.to('cuda')
print(f"✅ Model on {torch.cuda.get_device_name(0)}")

print("\nEncoding 936 statutes...")
statute_embeddings = model.encode(
    statute_texts_clean,
    batch_size=128,
    show_progress_bar=True,
    convert_to_tensor=True,
    device='cuda')
print(f"✅ {statute_embeddings.shape}")

print("\nEncoding 3,183 precedents...")
precedent_embeddings = model.encode(
    precedent_texts_clean,
    batch_size=128,
    show_progress_bar=True,
    convert_to_tensor=True,
    device='cuda')
print(f"✅ {precedent_embeddings.shape}")

print("\nEncoding 627 queries...")
query_embeddings = model.encode(
    query_texts_clean,
    batch_size=128,
    show_progress_bar=True,
    convert_to_tensor=True,
    device='cuda')
print(f"✅ {query_embeddings.shape}")

print("\n✅ ALL EMBEDDINGS READY!")

In [ ]:
# ================================
# CELL 6 — SBERT Retrieval
# ================================
from sentence_transformers import util
import numpy as np

def evaluate(all_retrieved, all_relevant, k):
    f1_scores  = []
    map_scores = []
    mrr_scores = []
    for retrieved, relevant in zip(
            all_retrieved, all_relevant):
        if len(relevant) == 0:
            continue
        # f1
        p = len(set(retrieved[:k]) &
                set(relevant)) / k
        r = len(set(retrieved[:k]) &
                set(relevant)) / len(relevant)
        f1 = 2*p*r/(p+r) if p+r > 0 else 0
        f1_scores.append(f1)
        # mrr
        for rank, doc_id in enumerate(
                retrieved, start=1):
            if doc_id in relevant:
                mrr_scores.append(1.0/rank)
                break
        else:
            mrr_scores.append(0.0)
    return {
        f'F1@{k}'  : np.mean(f1_scores)*100,
        'MRR'      : np.mean(mrr_scores)*100
    }

# LSR
print("SBERT retrieval for LSR...")
sbert_lsr_retrieved = []
for i, q_emb in enumerate(query_embeddings):
    if i % 100 == 0:
        print(f"  Query {i}/627...")
    scores = util.cos_sim(
        q_emb, statute_embeddings)[0]
    top_k = torch.topk(scores, k=10)
    retrieved_ids = [
        statute_ids[idx]
        for idx in top_k.indices.tolist()]
    sbert_lsr_retrieved.append(retrieved_ids)

# Evaluate LSR
best_f1 = 0
best_results_lsr = None
for eval_k in range(1, 11):
    results = evaluate(
        sbert_lsr_retrieved,
        gold_statutes, eval_k)
    if results[f'F1@{eval_k}'] > best_f1:
        best_f1          = results[f'F1@{eval_k}']
        best_results_lsr = results

print(f"\n✅ SBERT LSR Results:")
for m, v in best_results_lsr.items():
    print(f"  {m}: {v:.2f}%")

# PCR
print("\nSBERT retrieval for PCR...")
sbert_pcr_retrieved = []
for i, q_emb in enumerate(query_embeddings):
    if i % 100 == 0:
        print(f"  Query {i}/627...")
    scores = util.cos_sim(
        q_emb, precedent_embeddings)[0]
    top_k = torch.topk(scores, k=10)
    retrieved_ids = [
        precedent_ids[idx]
        for idx in top_k.indices.tolist()]
    sbert_pcr_retrieved.append(retrieved_ids)

# Evaluate PCR
best_f1 = 0
best_results_pcr = None
for eval_k in range(1, 11):
    results = evaluate(
        sbert_pcr_retrieved,
        gold_precedents, eval_k)
    if results[f'F1@{eval_k}'] > best_f1:
        best_f1          = results[f'F1@{eval_k}']
        best_results_pcr = results

print(f"\n✅ SBERT PCR Results:")
for m, v in best_results_pcr.items():
    print(f"  {m}: {v:.2f}%")

print("\n✅ SBERT COMPLETE!")

In [ ]:
# ================================
# CELL 7 — Save & Download
# ================================
import torch
from google.colab import files

torch.save(statute_embeddings,
           'statute_embeddings.pt')
torch.save(precedent_embeddings,
           'precedent_embeddings.pt')
torch.save(query_embeddings,
           'query_embeddings.pt')

print("✅ Saved!")
files.download('statute_embeddings.pt')
files.download('precedent_embeddings.pt')
files.download('query_embeddings.pt')
print("✅ All downloaded!")